# semantic chunking
Markdown header split
↓
Semantic merge nearby related chunks
↓
Final chunks
1. meaning is similar enough
2. merged chunk is not too large
3. both chunks belong to the same main topic

In [1]:
import os
import numpy as np
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document


file_path = "website.md"

if not os.path.exists(file_path):
    raise FileNotFoundError(f"File not found: {file_path}")

with open(file_path, "r", encoding="utf-8") as f:
    markdown_text = f.read()


# 1. Split by Markdown headers first
headers_to_split_on = [
    ("#", "main_topic"),
    ("##", "sub_topic"),
    ("###", "detail_level"),
]

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=False
)

header_chunks = markdown_splitter.split_text(markdown_text)

print("Header chunks:", len(header_chunks))


# 2. Embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)


# 3. Helper function for cosine similarity
def cosine_similarity(vec1, vec2):
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)

    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))


# 4. Prepare clean chunk texts
valid_chunks = []

for doc in header_chunks:
    text = doc.page_content.strip()

    if not text:
        continue

    valid_chunks.append(doc)


texts = [doc.page_content for doc in valid_chunks]


# 5. Create embeddings for each header chunk
chunk_embeddings = embedding_model.embed_documents(texts)


# 6. Semantic merge settings
SIMILARITY_THRESHOLD = 0.62
MAX_CHARS_PER_CHUNK = 1800
MIN_CHARS_PER_CHUNK = 300


semantic_chunks = []

current_text = valid_chunks[0].page_content.strip()
current_metadata = valid_chunks[0].metadata.copy()

for i in range(1, len(valid_chunks)):
    previous_embedding = chunk_embeddings[i - 1]
    current_embedding = chunk_embeddings[i]

    similarity = cosine_similarity(previous_embedding, current_embedding)

    next_text = valid_chunks[i].page_content.strip()
    merged_text = current_text + "\n\n" + next_text

    same_main_topic = (
        valid_chunks[i - 1].metadata.get("main_topic", "") ==
        valid_chunks[i].metadata.get("main_topic", "")
    )

    should_merge = (
        similarity >= SIMILARITY_THRESHOLD
        and len(merged_text) <= MAX_CHARS_PER_CHUNK
        and same_main_topic
    )

    if should_merge:
        current_text = merged_text
    else:
        if len(current_text) >= MIN_CHARS_PER_CHUNK:
            semantic_chunks.append(
                Document(
                    page_content=current_text,
                    metadata=current_metadata
                )
            )
        else:
            semantic_chunks.append(
                Document(
                    page_content=current_text,
                    metadata=current_metadata
                )
            )

        current_text = next_text
        current_metadata = valid_chunks[i].metadata.copy()


# Add last chunk
semantic_chunks.append(
    Document(
        page_content=current_text,
        metadata=current_metadata
    )
)


# 7. Add final metadata
final_chunks = []

for i, chunk in enumerate(semantic_chunks, start=1):
    metadata = {
        "chunk_id": i,
        "source": file_path,
        "main_topic": chunk.metadata.get("main_topic", "Website"),
        "sub_topic": chunk.metadata.get("sub_topic", ""),
        "detail_level": chunk.metadata.get("detail_level", "")
    }

    final_chunks.append(
        Document(
            page_content=chunk.page_content,
            metadata=metadata
        )
    )


print("Final semantic chunks:", len(final_chunks))


# 8. Preview chunks
for chunk in final_chunks:
    print(f"\n--- Chunk {chunk.metadata['chunk_id']} ---")
    print("Characters:", len(chunk.page_content))
    print("Metadata:", chunk.metadata)
    print(chunk.page_content[:1000])

/home/puresitedesktop/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Header chunks: 120


Loading weights: 100%|██████████████████████| 199/199 [00:00<00:00, 1622.52it/s]


Final semantic chunks: 53

--- Chunk 1 ---
Characters: 229
Metadata: {'chunk_id': 1, 'source': 'website.md', 'main_topic': 'You Have Arrived In The Hub Of Creativity And Innovation.', 'sub_topic': '', 'detail_level': ''}
# You Have Arrived In The Hub Of Creativity And Innovation.
At Dollarbird, we strive to build connections based on reliability and trust. For any questions, we're also accessible around the clock. You just need to reach out!
----

--- Chunk 2 ---
Characters: 332
Metadata: {'chunk_id': 2, 'source': 'website.md', 'main_topic': 'Our Identity', 'sub_topic': '', 'detail_level': ''}
# Our Identity
You can count on us to help you achieve your goals and find solutions to your company's problems. With our team of specialists in Advertising, Software Development, Design, and Sales Support, we provide end-to-end solutions to all your needs. We can provide an ultimate solution by listening to your pain points.
----

--- Chunk 3 ---
Characters: 553
Metadata: {'chunk_id': 3, 'source

In [2]:
import requests

OLLAMA_URL = "http://localhost:11434/api/embeddings"
EMBEDDING_MODEL = "bge-m3:567m"


def get_embedding(text):
    response = requests.post(
        OLLAMA_URL,
        json={
            "model": EMBEDDING_MODEL,
            "prompt": text
        }
    )

    if response.status_code != 200:
        raise Exception(f"Ollama embedding error: {response.text}")

    return response.json()["embedding"]


embeddings = []

for chunk in final_chunks:
    chunk_text = chunk.page_content

    embedding = get_embedding(chunk_text)

    if len(embedding) != 1024:
        raise ValueError(f"Wrong embedding dimension: {len(embedding)}")

    embeddings.append({
        "chunk_id": chunk.metadata["chunk_id"],
        "text": chunk_text,
        "embedding": embedding,
        "metadata": {
            "source": chunk.metadata.get("source", ""),
            "main_topic": chunk.metadata.get("main_topic", ""),
            "sub_topic": chunk.metadata.get("sub_topic", ""),
            "detail_level": chunk.metadata.get("detail_level", "")
        }
    })

print("Total embeddings created:", len(embeddings))
print("Embedding dimension:", len(embeddings[0]["embedding"]))

Total embeddings created: 53
Embedding dimension: 1024


In [3]:
import chromadb

client = chromadb.PersistentClient(path="./chroma_db_semantic")

collection = client.get_or_create_collection(
    name="website_embeddings_semantic2"
)

ids = []
documents = []
vectors = []
metadatas = []

for item in embeddings:
    ids.append(str(item["chunk_id"]))
    documents.append(item["text"])
    vectors.append(item["embedding"])
    metadatas.append({
        "chunk_id": item["chunk_id"],
        "source": item["metadata"]["source"],
        "main_topic": item["metadata"]["main_topic"],
        "sub_topic": item["metadata"]["sub_topic"],
        "detail_level": item["metadata"]["detail_level"]
    })

collection.upsert(
    ids=ids,
    documents=documents,
    embeddings=vectors,
    metadatas=metadatas
)

print("Stored chunks in ChromaDB:", collection.count())

Stored chunks in ChromaDB: 53
